# Notebook B: is the count signal actually *used*? Patching and steering LLaVA (GPU, about 1-1.5 h)

Notebook A showed the model builds a clean linear count signal at layers 8-16 and that the grating signal is about half as strong from layer 14 on when history is in the prompt. It cannot show whether the model *uses* the count signal. Here we intervene on the residual stream at the answer position (last prompt token).

**B1. Patch history state into a no-history run** (same image). At layer $L$ we overwrite the no-history residual stream at the last position with the cached history-run one. *Recovery* at layer $L$ is the fraction of the history-induced shift in $\Delta$ that comes back. It tells us from which layer on the count information sits at the answer position (if recovery is already ~1, later layers no longer need the history tokens).
**B1r. Reverse:** put the no-history state into a history run; the fraction of the shift *removed*.
**B2. Steer** a no-history run by adding $\pm$ a direction (norm 1x and 4x the natural half-jump of the count signal) at layer $L$. Directions: the *shift* direction (mean history-minus-no-history difference, high vs low count), the *count* direction from A, the grating-side *evidence* direction (positive control: should move $\Delta$ a lot), a *shuffled-label* direction and a *random* direction (negative controls).
**B0. Which wording hurts the grating, and which still carries the prior?** The quick test showed a neutral sentence of the same length does *not* hurt, the count sentence does (swing ratio 0.44), a shorter form less (0.71), and putting it before the image is worst (0.20). So the loss is about the content (the words left/right and numbers), not length or position. B0 tries four wordings (sentence, short, letters L/R, sequence of letters), each at 2, 5 and 8 of 10 right, and reports swing ratio and prior effect for each, to look for a wording that keeps the prior and loses less evidence.
**C. Interference.** In the same runs: does steering or patching also change the **swing** (right-grating minus left-grating $\Delta$)? If the count direction is also what weakens the evidence, adding it to a no-history run should shrink the swing.

**Reading the result.** At late layers the count and evidence directions have cosine ~0.7 (both point along the answer axis), so a steering effect there is not diagnostic. The informative layers are 8-12, where cosine is ~0: if adding the count direction there moves $\Delta$, later layers read it.

**Setup:** GPU on, Internet on (model download), dataset `llava-pve-outputs-run` attached. It recomputes the trials from `SEED` and the directions from the cache, so no other files are needed. Every stage checkpoints to `/kaggle/working/B_out` and resumes if the session dies; use *Save & Run All (Commit)*.
**This notebook was tested on a mock model, not on the real LLaVA. The self-test cell stops with a clear message if a hook is wrong.**

I've sent the new notebook B with `QUICK = False`. I tested it only with a mock model, where the self-tests pass and every cell runs. I have not run it on the real LLaVA, so the first real check is its own self-tests when you start it.

**What changed after your quick run:**
- **Steering sizes:** steering at the natural size of the count signal did nothing at layers 12 and below, because the signal is tiny there. Steering now runs at 1× and 4× the natural size, at layers 12, 14, 16, 20 and 24. The evidence direction is still the positive control.
- **Self-test 5:** it now adds the evidence direction at layer 20, which is on the second GPU. That also checks the cross-device hook you hit before.
- **Prompt-wording screen (B0):**
  - **Why this screen:** your quick run showed that a neutral sentence does not hurt the grating swing, but the count sentence does. A shorter form hurt less, and putting the count before the image hurt most. So the loss comes from the content of the count, not its length or position.
  - **What it tests:** four wordings (sentence, short, letters L/R, and a sequence of letters), each at 2, 5 and 8 of 10 "right", plus the controls.
  - **What it reports:** for each wording, how much of the swing survives and how big the prior effect is. The goal is a wording that keeps the prior and loses less evidence.
- **Patching:** layers 8 to 28.
- **Resume:** B0 is now checkpointed too.

**To run it:** Save & Run All with 2 GPUs and the `KAGGLE_KEY` secret set. Expect about 1.5 to 2 hours. Checkpoints push to `ponishbhatia3/llava-pve-b-ckpt` every 10 minutes, so a crash can resume. Send me `B_out.zip` when it finishes.

**What I'll look at first:**
- **B0:** swing ratio and prior gap per wording.
- **Patching:** the recovery curve (how much of the count effect comes back at each layer), expected to start around layer 16.
- **Steering:** whether the count direction beats the shuffled and random directions at 4× size.

In [1]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~5 min): fewer trials and layers
SEED = 0                     # must match the main run (the trials are regenerated from it)
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
OUT = "/kaggle/working/B_out"
N_EXT, N_REST = 110, 50                      # patch trials: with extreme counts (<=2 or >=8 'right'), and the rest
N_STEER = 40                                 # steering images (balanced left/right, contrast >= 0.0625)
B0_N = 80                                    # images for the prompt-wording screen (half left, half right)
PATCH_LAYERS = [8, 10, 12, 14, 16, 18, 20, 24, 28]
STEER_LAYERS = [12, 14, 16, 20, 24]          # below ~12 the natural count signal is ~0 in norm, so steering at natural size cannot do anything there
AMPS = [1, 4]                                # steering size as a multiple of the natural half-jump norm
DIRS = ["shift", "prior", "evid", "shuf", "rand1"]
DO_REVERSE = True
if QUICK:
    N_EXT, N_REST, N_STEER, B0_N, PATCH_LAYERS, STEER_LAYERS, DIRS = 24, 8, 16, 16, [8, 12, 16, 20], [14, 20], ["shift", "evid", "rand1"]
import os, re, glob, json, math, time, zipfile, shutil, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [2]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-b-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/B_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_EXT=N_EXT, N_REST=N_REST, N_STEER=N_STEER, PATCH_LAYERS=PATCH_LAYERS, STEER_LAYERS=STEER_LAYERS, DIRS=DIRS, DO_REVERSE=DO_REVERSE)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook B (patching/steering). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE B checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("patch_rows.csv", "steer_rows.csv", "b0_rows.csv"):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (preflight) | server says: Ok
checkpoint push works; later pushes happen every 10 min and at the end of each stage


In [3]:
# ===================== trials, prompts, images: identical to the main run (counts10 history) =====================
CONTRASTS = [0.0, 0.03, 0.045, 0.0625, 0.09, 0.125, 1.0]; N_FULL, FULL_UNBIASED, WINDOW = 600, 90, 10
SIZE = 336; _Y, _X = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)
def make_trials(n, seed, contrasts, n_unbiased):
    rng = np.random.default_rng(seed); p = np.full(n, 0.5)
    t, cur = n_unbiased, float(rng.choice([0.2, 0.8]))
    while t < n:
        L = 20 + int(rng.exponential(60))
        while L > 100: L = 20 + int(rng.exponential(60))
        p[t:t + L] = cur; t, cur = t + L, round(1.0 - cur, 2)
    side = np.where(rng.random(n) < p, "right", "left"); c = rng.choice(contrasts, n)
    return dict(n=n, p_right=p, side=side, contrast=c, dx=rng.uniform(-15, 15, n), dy=rng.uniform(-30, 30, n), phi=rng.uniform(0, 2 * np.pi, n))
def render(T, t, seed):
    cx = SIZE / 2 + (84 if T["side"][t] == "right" else -84) + T["dx"][t]; cy = SIZE / 2 + T["dy"][t]
    env = np.exp(-((_X - cx) ** 2 + (_Y - cy) ** 2) / (2 * 36 ** 2))
    a = 0.5 + 0.5 * T["contrast"][t] * env * np.sin(2 * np.pi * (_X - cx) / 24 + T["phi"][t])
    a = a + np.random.default_rng(seed * 100003 + t).normal(0, 0.02, a.shape)
    return Image.fromarray(np.stack([(np.clip(a, 0, 1) * 255).astype(np.uint8)] * 3, -1))
def hist_counts(T, t, w):
    past = T["side"][max(0, t - w):t]; return len(past), int((past == "right").sum())
def history_text(T, t):
    n, nr = hist_counts(T, t, WINDOW)
    return "No previous trials." if n == 0 else f"In the last {n} trials the grating was on the left {n - nr} times and on the right {nr} times."
QS = ["Is the grating on the left or right of the image? Answer with one word.", "Is the grating on the right or left of the image? Answer with one word."]
def prompt(hist, q): return f"USER: <image>\n{hist}\n{q} ASSISTANT:" if hist else f"USER: <image>\n{q} ASSISTANT:"
T = make_trials(N_FULL, SEED + 1, CONTRASTS, FULL_UNBIASED)

In [4]:
# ===================== cache + directions (recomputed from the cached residual streams, same recipe as notebook A) =====================
ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])    # (trial, [none,hist], layer 0..32, dim)
NTR, _, NLAY, D = HS.shape
w_ = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w_.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["d_none"], M["d_hist"] = w_["none"], w_["hist"]; M["shift"] = M.d_hist - M.d_none
assert (M.side.values == T["side"]).all() and np.allclose(M.contrast.values, T["contrast"]), "regenerated trials differ from the main run: check SEED / CONTRASTS"
NR = M.n_hist_right.values.astype(int); FULL = (M.n_hist.values == 10)
frac = (NR + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac))
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values
lsq = lambda A, B: np.linalg.lstsq(A, B, rcond=None)[0]
def count_dir(X, target, Cc):
    Xr = X - Cc @ lsq(Cc, X); yt = target - Cc @ lsq(Cc, target); d = Xr.T @ yt; return d / (np.linalg.norm(d) + 1e-12)
hi_c = M.contrast.values >= 0.09; sr = M.side.values == "right"
hi_cnt, lo_cnt = FULL & (NR >= 8), FULL & (NR <= 2)
rng0 = np.random.default_rng(0); perm = rng0.permutation(FULL.sum())
DIRVEC, NORM, RESNORM = {}, np.zeros(NLAY), np.zeros(NLAY)
for name in DIRS: DIRVEC[name] = np.zeros((NLAY, D), np.float32)
UNIT = {name: np.zeros((NLAY, D), np.float32) for name in ["shift", "prior", "evid", "shuf", "rand1", "rand2"]}    # unit-norm versions, always computed
for L in range(1, NLAY):
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32)
    shift = ((Xh - Xn)[hi_cnt].mean(0) - (Xh - Xn)[lo_cnt].mean(0)) / 2          # natural half-jump from a no-history state to a high/low-count state
    NORM[L] = np.linalg.norm(shift); RESNORM[L] = np.linalg.norm(Xn, axis=1).mean(); unit = lambda v: v / (np.linalg.norm(v) + 1e-12)
    cand = {"shift": unit(shift),
            "prior": count_dir(Xh[FULL], TP[FULL], cell[FULL]),
            "evid": unit(Xn[hi_c & sr].mean(0) - Xn[hi_c & ~sr].mean(0)),
            "shuf": count_dir(Xh[FULL], TP[FULL][perm], cell[FULL]),
            "rand1": unit(np.random.default_rng(11 + L).normal(size=D)), "rand2": unit(np.random.default_rng(97 + L).normal(size=D))}
    for name in cand: UNIT[name][L] = cand[name]
    for name in DIRS: DIRVEC[name][L] = cand[name] * NORM[L]                     # every direction gets the SAME norm
nat_hi = M["shift"][hi_cnt].mean(); nat_lo = M["shift"][lo_cnt].mean(); NAT_HALF = (nat_hi - nat_lo) / 2
print(f"cache {HS.shape} | natural behavioral half-jump in delta (high-count minus low-count)/2 = {NAT_HALF:.2f} logits")
print("natural half-jump norm | typical residual norm, per layer:", {L: (round(float(NORM[L]), 2), round(float(RESNORM[L]), 1)) for L in (8, 10, 12, 14, 16, 20, 24)})
print("cos(count direction, evidence direction) at layers 8/12/16/20:", {L: round(float(UNIT['prior'][L] @ UNIT['evid'][L]), 2) for L in (8, 12, 16, 20)})

cache (600, 2, 33, 4096) | natural behavioral half-jump in delta (high-count minus low-count)/2 = 0.33 logits
natural half-jump norm | typical residual norm, per layer: {8: (0.05, 9.3), 10: (0.11, 12.8), 12: (0.61, 17.1), 14: (1.09, 20.7), 16: (1.87, 27.3), 20: (3.18, 54.2), 24: (3.66, 87.6)}
cos(count direction, evidence direction) at layers 8/12/16/20: {8: -0.01, 12: 0.06, 16: 0.42, 20: 0.69}


In [5]:
# ===================== model runner =====================
if MOCK:
    class Runner:
        """Numpy stand-in: state at layer L is linear in (evidence, count); delta reads it out. Pipeline test only."""
        def __init__(s): s.rng = np.random.default_rng(5)
        def prep(s, img, text):
            a = np.abs(np.asarray(img)[..., 0].astype(float) - 128); cx = (a.sum(0) * np.arange(SIZE)).sum() / (a.sum() + 1e-6)
            m = re.search(r"on the right (\d+) times", text); n = None
            if m: nr = int(m.group(1)); nl = int(re.search(r"on the left (\d+) times", text).group(1)); n = (nr + .5) / (nr + nl + 1)
            return dict(ev=float(np.clip((cx - 168) * 0.03, -1.7, 1.7)), frac=n)
        def state(s, inp, L):
            x = np.zeros(D, np.float32); h = np.random.default_rng(int(abs(inp["ev"]) * 1e4) % 99991)
            x += 0.02 * h.normal(size=D)
            if L >= 4: x[0] += inp["ev"] * (0.5 if inp["frac"] is not None else 1.0)
            if L >= 8 and inp["frac"] is not None: x[1] += (inp["frac"] - .5) * 3; x[0] += (inp["frac"] - .5) * 0.5
            return x
        def readout(s, x): return -0.7 + 3.0 * x[0] + 1.0 * x[1]
        def delta(s, inp, edit=None):
            x = s.state(inp, 32)
            if edit is not None:
                L, mode, vec = edit; xl = s.state(inp, L)
                x = x + ((vec - xl) if mode == "replace" else vec)
            return float(s.readout(x))
        def hs(s, inp): return np.stack([s.state(inp, L) for L in range(NLAY)]).astype(np.float16)
    RUN = Runner()
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto",
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    L_ID, R_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("Left", "Right")]
    _lists = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == 32]
    assert len(_lists) == 1, f"expected exactly one 32-layer list, found {[n for n, _ in _lists]}"
    LAYERS = _lists[0][1]; print("language-model layers found at:", _lists[0][0])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp, edit=None):
            handle = None
            if edit is not None:
                L, mode, vec = edit; v0 = torch.tensor(vec, dtype=torch.float16)
                def hook(mod, args, output):                      # output of decoder layer L-1 == hidden_states[L]
                    h = output[0] if isinstance(output, tuple) else output
                    v = v0.to(h.device, h.dtype)                  # the model can be split over 2 GPUs: put the vector where this layer lives
                    h = h.clone()
                    if mode == "replace": h[:, -1, :] = v
                    else: h[:, -1, :] = h[:, -1, :] + v
                    return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
                handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally:
                if handle is not None: handle.remove()
            return float((lg[R_ID] - lg[L_ID]).item())
        def hs(s, inp):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            return torch.stack([h[0, -1] for h in out.hidden_states]).float().cpu().numpy().astype(np.float16)
        def captured(s, inp, L):                                  # what the hook sees at layer L (for the self-test)
            box = {}
            def hook(mod, args, output): box["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
            hd = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): model(**inp)
            finally: hd.remove()
            return box["h"]
    RUN = Runner()
print("runner ready")

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

language-model layers found at: model.language_model.layers
runner ready


In [6]:
# ===================== SELF-TEST: stop here if anything is off =====================
def inputs(t, with_hist, q=0):
    return RUN.prep(render(T, t, SEED + 1), prompt(history_text(T, t) if with_hist else None, QS[q]))
ok = np.where(FULL & (M.contrast.values >= 0.09))[0]; test_ts = list(ok[:6])
res = []
for t in test_ts:
    dn = (RUN.delta(inputs(t, False, 0)) + RUN.delta(inputs(t, False, 1))) / 2; dh = (RUN.delta(inputs(t, True, 0)) + RUN.delta(inputs(t, True, 1))) / 2
    res.append((dn - M.d_none.values[t], dh - M.d_hist.values[t]))
res = np.array(res); say(f"[self-test 1] reproduction of the cached delta (new - cached): none max|diff| {np.abs(res[:, 0]).max():.3f}, hist max|diff| {np.abs(res[:, 1]).max():.3f}")
assert np.abs(res).max() < 0.35, "cannot reproduce the cached deltas: trials, prompts or model setup differ from the main run"
t = test_ts[0]; inp = inputs(t, True, 0); d0 = RUN.delta(inp)
d_id = RUN.delta(inp, edit=(12, "add", np.zeros(D, np.float32))); say(f"[self-test 2] adding a zero vector changes delta by {abs(d_id - d0):.4f} (should be ~0)")
assert abs(d_id - d0) < 0.05
own = HS[t, 1, 12, :].astype(np.float32); d_own = RUN.delta(inp, edit=(12, "replace", own)); say(f"[self-test 3] replacing the state with the history run's own cached state changes delta by {abs(d_own - d0):.4f} (should be small)")
assert abs(d_own - d0) < 0.25
if not MOCK:
    cap = RUN.captured(inp, 12); cache = HS[t, 1, 12, :].astype(np.float32)
    cs = float(cap @ cache / (np.linalg.norm(cap) * np.linalg.norm(cache))); say(f"[self-test 4] hook state vs cached hidden_states[12]: cosine {cs:.4f} (should be ~1)")
    assert cs > 0.98, "hook indexing does not match hidden_states: layer off by one?"
Lt = 20                                                  # layer 20 lives on the SECOND GPU: this also tests the cross-device 'add' hook
big_vec = (UNIT["evid"][Lt] * 0.5 * RESNORM[Lt]).astype(np.float32); big = RUN.delta(inp, edit=(Lt, "add", big_vec))
say(f"[self-test 5] adding the evidence direction (half the typical residual norm) at layer {Lt} moves delta by {big - d0:+.2f} (should be clearly positive)")
assert big - d0 > (0.01 if MOCK else 0.5), "steering along the evidence direction does nothing: the 'add' hook is broken"
say("self-tests passed")

[self-test 1] reproduction of the cached delta (new - cached): none max|diff| 0.000, hist max|diff| 0.000
[self-test 2] adding a zero vector changes delta by 0.0000 (should be ~0)
[self-test 3] replacing the state with the history run's own cached state changes delta by 0.0000 (should be small)
[self-test 4] hook state vs cached hidden_states[12]: cosine 1.0000 (should be ~1)
[self-test 5] adding the evidence direction (half the typical residual norm) at layer 20 moves delta by +4.30 (should be clearly positive)
self-tests passed


In [7]:
# ===================== B0: which prompt wording hurts the grating, and which still carries the prior? (~15 min, checkpointed) =====================
# Quick-test finding: a neutral sentence of the same length does NOT hurt (ratio ~1.1), but the count sentence does (~0.44), the short form less (~0.71).
# So the loss is about the CONTENT (words 'left'/'right' + numbers), not length or position. Here: other wordings, each at 2, 5 and 8 of 10 'right'.
rngb = np.random.default_rng(3); okb = np.where(M.contrast.values >= 0.09)[0]
B0_T = np.sort(np.r_[rngb.choice(okb[M.side.values[okb] == "right"], B0_N // 2, replace=False), rngb.choice(okb[M.side.values[okb] == "left"], B0_N // 2, replace=False)])
def _seq(t, k): return " ".join(np.random.default_rng(1000 + t).permutation(np.array(["R"] * k + ["L"] * (10 - k))))
FMT = {"sentence": lambda t, k: f"In the last 10 trials the grating was on the left {10 - k} times and on the right {k} times.",
       "short":    lambda t, k: f"Recent trials: {10 - k} left, {k} right.",
       "letters":  lambda t, k: f"Recent trials: {10 - k} L, {k} R.",
       "sequence": lambda t, k: f"Recent trials: {_seq(t, k)}."}
KS = (2, 5, 8)
VARIANTS = ["none", "neutral", "sentence_before_image_5"] + [f"{f}_{k}" for f in FMT for k in KS]
def b0_prompt(name, t, q):
    if name == "none": return prompt(None, q)
    if name == "neutral": return prompt("The picture was taken with a plain camera in a quiet room on a sunny afternoon.", q)
    if name == "sentence_before_image_5": return f"USER: {FMT['sentence'](t, 5)}\n<image>\n{q} ASSISTANT:"
    f, k = name.rsplit("_", 1); return prompt(FMT[f](t, int(k)), q)
b0p = f"{OUT}/b0_rows.csv"; d0_ = pd.read_csv(b0p) if os.path.exists(b0p) else pd.DataFrame()
have0 = set(d0_.t) if len(d0_) else set(); b0 = d0_.to_dict("records") if len(d0_) else []
todo0 = [t for t in B0_T if t not in have0]; t0 = time.time()
print(f"B0: {len(B0_T)} images x {len(VARIANTS)} prompts = {len(B0_T) * len(VARIANTS)} forwards; images left: {len(todo0)}")
for i, t in enumerate(todo0):
    img = render(T, t, SEED + 1)
    for name in VARIANTS: b0.append(dict(t=t, side=M.side.values[t], variant=name, delta=RUN.delta(RUN.prep(img, b0_prompt(name, t, QS[0])))))
    if (i + 1) % 8 == 0 or i == len(todo0) - 1:
        pd.DataFrame(b0).to_csv(b0p, index=False); maybe_push("B0"); el = time.time() - t0
        print(f"  {i+1}/{len(todo0)} | {el/60:.1f} min | ~{el/(i+1)*(len(todo0)-i-1)/60:.0f} min left", flush=True)
B0 = pd.DataFrame(b0); B0.to_csv(b0p, index=False); maybe_push("B0 done", force=True)

B0: 80 images x 15 prompts = 1200 forwards; images left: 80
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (B0) | server says: Ok
  8/80 | 1.2 min | ~11 min left
  16/80 | 2.5 min | ~10 min left
  24/80 | 3.9 min | ~9 min left
  32/80 | 5.3 min | ~8 min left
  40/80 | 6.6 min | ~7 min left
  48/80 | 7.9 min | ~5 min left
  56/80 | 9.3 min | ~4 min left
  64/80 | 10.6 min | ~3 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (B0) | server says: Ok
  72/80 | 12.0 min | ~1 min left
  80/80 | 13.4 min | ~0 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (B0 done) | server says: Ok


In [8]:
# ===================== B0 analysis =====================
Pv = B0.pivot(index="t", columns="variant", values="delta").loc[B0_T]
sd = M.side.values[Pv.index.values]; ri, li = np.where(sd == "right")[0], np.where(sd == "left")[0]
def sw(col, rr, ll): v = Pv[col].values; return v[rr].mean() - v[ll].mean()
def bci(fn, n=300, seed=0):
    r = np.random.default_rng(seed); vals = [fn(r.choice(ri, len(ri)), r.choice(li, len(li))) for _ in range(n)]; return np.percentile(vals, [2.5, 97.5])
sw_none = sw("none", ri, li)
say(f"\n[B0] swing (right minus left delta, contrast >= 0.09, {len(B0_T)} images); no-history swing = {sw_none:.2f}. ratio = swing / no-history swing (1 = no loss)")
rows_b0 = []
for name in VARIANTS:
    lo, hi = bci(lambda rr, ll: sw(name, rr, ll) / sw("none", rr, ll))
    rows_b0.append(dict(variant=name, swing=sw(name, ri, li), ratio=sw(name, ri, li) / sw_none, lo=lo, hi=hi, mean_delta=Pv[name].mean()))
B0S = pd.DataFrame(rows_b0); B0S.to_csv(f"{OUT}/b0_summary.csv", index=False); say(B0S.round(2).to_string(index=False))
say("\nprior effect per wording = mean over images of delta(8 of 10 right) - delta(2 of 10 right), logits [95% CI]; compare the LLaVA main run (counts10 sentence): about +0.66 for high minus low count")
for f in FMT:
    gap = (Pv[f"{f}_8"] - Pv[f"{f}_2"]).values; r = np.random.default_rng(0); bs = [gap[r.integers(0, len(gap), len(gap))].mean() for _ in range(300)]
    say(f"  {f:9s} prior gap {gap.mean():+.2f} [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}] | swing ratio at 5 of 10: {B0S[B0S.variant == f + '_5'].ratio.iloc[0]:.2f}")


[B0] swing (right minus left delta, contrast >= 0.09, 80 images); no-history swing = 3.33. ratio = swing / no-history swing (1 = no loss)
                variant  swing  ratio   lo   hi  mean_delta
                   none   3.33   1.00 1.00 1.00       -0.78
                neutral   3.90   1.17 1.14 1.20       -0.77
sentence_before_image_5   0.68   0.20 0.18 0.22       -1.42
             sentence_2   1.60   0.48 0.46 0.49       -0.61
             sentence_5   1.35   0.40 0.39 0.42       -0.33
             sentence_8   1.23   0.37 0.36 0.38        0.25
                short_2   2.47   0.74 0.73 0.76       -0.44
                short_5   2.52   0.75 0.73 0.78       -0.31
                short_8   2.16   0.65 0.63 0.67       -0.11
              letters_2   2.35   0.71 0.69 0.72       -0.69
              letters_5   2.32   0.70 0.68 0.71       -0.56
              letters_8   2.43   0.73 0.71 0.75       -0.40
             sequence_2   2.74   0.82 0.80 0.85       -0.61
             sequence

In [9]:
# ===================== B1 / B1r: PATCHING (checkpointed) =====================
rng = np.random.default_rng(1)
ext = np.where(FULL & ((NR >= 8) | (NR <= 2)))[0]; rest = np.where(FULL & ~((NR >= 8) | (NR <= 2)))[0]
PATCH_T = np.sort(np.r_[rng.choice(ext, min(N_EXT, len(ext)), replace=False), rng.choice(rest, min(N_REST, len(rest)), replace=False)])
pp = f"{OUT}/patch_rows.csv"; done = pd.read_csv(pp) if os.path.exists(pp) else pd.DataFrame()
have = set(done.t) if len(done) else set(); rows = done.to_dict("records") if len(done) else []
t0 = time.time(); todo = [t for t in PATCH_T if t not in have]; print(f"patch trials: {len(PATCH_T)} | to do: {len(todo)} | forwards per trial: {2 + len(PATCH_LAYERS) * (2 if DO_REVERSE else 1)}")
for i, t in enumerate(todo):
    inn, inh = inputs(t, False, 0), inputs(t, True, 0)
    rec = dict(t=t, side=M.side.values[t], contrast=M.contrast.values[t], nr=int(NR[t]), d_none=RUN.delta(inn), d_hist=RUN.delta(inh))
    for L in PATCH_LAYERS:
        rec[f"fwd_{L}"] = RUN.delta(inn, edit=(L, "replace", HS[t, 1, L, :].astype(np.float32)))              # history state -> no-history run
        if DO_REVERSE: rec[f"rev_{L}"] = RUN.delta(inh, edit=(L, "replace", HS[t, 0, L, :].astype(np.float32)))   # no-history state -> history run
    rows.append(rec)
    if (i + 1) % 10 == 0 or i == len(todo) - 1:
        pd.DataFrame(rows).to_csv(pp, index=False); maybe_push('patching'); el = time.time() - t0
        print(f"  {i+1}/{len(todo)} | {el/60:.1f} min | ~{el/(i+1)*(len(todo)-i-1)/60:.0f} min left", flush=True)
PT = pd.DataFrame(rows).sort_values("t").reset_index(drop=True); PT.to_csv(pp, index=False); maybe_push('patching done', force=True)

patch trials: 160 | to do: 160 | forwards per trial: 20
  10/160 | 2.2 min | ~33 min left
  20/160 | 4.5 min | ~31 min left
  30/160 | 6.7 min | ~29 min left
  40/160 | 8.9 min | ~27 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (patching) | server says: Ok
  50/160 | 11.2 min | ~25 min left
  60/160 | 13.5 min | ~22 min left
  70/160 | 15.7 min | ~20 min left
  80/160 | 17.9 min | ~18 min left
  90/160 | 20.2 min | ~16 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (patching) | server says: Ok
  100/160 | 22.5 min | ~13 min left
  110/160 | 24.7 min | ~11 min left
  120/160 | 27.0 min | ~9 min left
  130/160 | 29.2 min | ~7 min left
  140/160 | 31.4 min | ~4 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (patching) | server says: Ok
  150/160 | 33.7 min | ~2 min left
  160/160 | 36.0 min | ~0 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (patching done) | server says: Ok


In [10]:
# ===================== B1 analysis =====================
e = PT[(PT.nr >= 8) | (PT.nr <= 2)].copy(); e["sgn"] = np.where(e.nr >= 8, 1.0, -1.0)
def ratio(df, col, base="d_none", full="d_hist", sub=False):
    num = (df.sgn * (df[col] - df[base])).mean() if not sub else (df.sgn * (df[full] - df[col])).mean()
    den = (df.sgn * (df[full] - df[base])).mean(); return num / den
def boot_ratio(df, col, sub=False, n=300, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        b = df.iloc[r.integers(0, len(df), len(df))]; v.append(ratio(b, col, sub=sub))
    return np.percentile(v, [2.5, 97.5])
rows_p = []
for L in PATCH_LAYERS:
    rec = dict(layer=L, recovery=ratio(e, f"fwd_{L}")); rec["rec_lo"], rec["rec_hi"] = boot_ratio(e, f"fwd_{L}")
    if DO_REVERSE: rec["removed"] = ratio(e, f"rev_{L}", sub=True); rec["rem_lo"], rec["rem_hi"] = boot_ratio(e, f"rev_{L}", sub=True)
    rows_p.append(rec)
P1 = pd.DataFrame(rows_p); P1.to_csv(f"{OUT}/b1_patch_summary.csv", index=False)
say(f"\n[B1] patching. trials with extreme counts: {len(e)}; mean history-induced shift (sign-aligned) = {(e.sgn * (e.d_hist - e.d_none)).mean():+.2f} logits")
say("recovery = fraction of the shift that returns when the history state replaces the no-history state at layer L (1 = all of it)")
say("removed  = fraction of the shift erased when the no-history state replaces the history state at layer L")
say(P1.round(2).to_string(index=False))
# swing (right minus left delta at contrast >= 0.09) under patching
hc = PT[PT.contrast >= 0.09]
def swing(df, col): return df[df.side == "right"][col].mean() - df[df.side == "left"][col].mean()
sw = dict(none=swing(hc, "d_none"), hist=swing(hc, "d_hist")); sw.update({f"fwd_{L}": swing(hc, f"fwd_{L}") for L in PATCH_LAYERS})
say(f"\n[C, patching] swing (contrast >= 0.09, n={len(hc)}): none {sw['none']:.2f} | hist {sw['hist']:.2f} (ratio {sw['hist']/sw['none']:.2f})")
say("  swing after patching the history state into the no-history run, ratio to none: " + ", ".join(f"L{L}: {sw[f'fwd_{L}']/sw['none']:.2f}" for L in PATCH_LAYERS))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].errorbar(P1.layer, P1.recovery, yerr=[P1.recovery - P1.rec_lo, P1.rec_hi - P1.recovery], marker="o", label="recovery (history -> none)")
if DO_REVERSE: ax[0].errorbar(P1.layer + .3, P1.removed, yerr=[P1.removed - P1.rem_lo, P1.rem_hi - P1.removed], marker="s", label="removed (none -> history)")
ax[0].axhline(1, c="gray", lw=.5); ax[0].axhline(0, c="gray", lw=.5); ax[0].set_xlabel("layer"); ax[0].set_title("B1: fraction of the prior shift carried by the last-position state"); ax[0].legend(fontsize=8)
ax[1].plot(PATCH_LAYERS, [sw[f"fwd_{L}"] / sw["none"] for L in PATCH_LAYERS], "o-", label="patched run"); ax[1].axhline(sw["hist"] / sw["none"], c="r", ls="--", label="history run"); ax[1].axhline(1, c="gray", lw=.5)
ax[1].set_xlabel("layer"); ax[1].set_title("C: swing ratio after patching (1 = no suppression)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/b1_patch.png", dpi=130); plt.close()


[B1] patching. trials with extreme counts: 110; mean history-induced shift (sign-aligned) = +0.16 logits
recovery = fraction of the shift that returns when the history state replaces the no-history state at layer L (1 = all of it)
removed  = fraction of the shift erased when the no-history state replaces the history state at layer L
 layer  recovery  rec_lo  rec_hi  removed  rem_lo  rem_hi
     8     -0.04   -0.23    0.06     0.19   -0.93    1.53
    10     -0.08   -0.85    0.50     0.25   -0.58    1.24
    12     -0.32   -2.40    1.03     0.17   -0.03    0.40
    14     -0.99   -8.50    4.53    -0.05   -1.56    0.98
    16      0.11   -2.31    1.70     0.29   -0.84    1.22
    18      1.30   -0.73    3.86     1.19   -1.55    3.92
    20      1.36   -0.13    3.42     1.15    0.20    2.49
    24      1.21    0.20    2.59     1.07    0.51    1.95
    28      1.27    0.12    2.81     1.08    0.61    1.83

[C, patching] swing (contrast >= 0.09, n=83): none 3.17 | hist 1.84 (ratio 0.58)
  

In [11]:
# ===================== B2 / C: STEERING a no-history run (checkpointed) =====================
# size = AMPS x the natural half-jump norm (the size of the count signal itself); both signs; the evidence direction is the positive control
rng = np.random.default_rng(2)
okc = np.where((M.contrast.values >= 0.0625))[0]
STEER_T = np.sort(np.r_[rng.choice(okc[M.side.values[okc] == "right"], N_STEER // 2, replace=False), rng.choice(okc[M.side.values[okc] == "left"], N_STEER // 2, replace=False)])
sp = f"{OUT}/steer_rows.csv"; sdone = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
srows = sdone.to_dict("records") if len(sdone) else []; seen = set(zip(sdone.layer, sdone.dir, sdone.s)) if len(sdone) else set()
base_inp = {t: inputs(t, False, 0) for t in STEER_T}
if (0, "base", 0) not in seen:
    for t in STEER_T: srows.append(dict(layer=0, dir="base", s=0, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t])))
    pd.DataFrame(srows).to_csv(sp, index=False)
combos = [(L, d, s) for L in STEER_LAYERS for d in DIRS for a in AMPS for s in (-a, a) if (L, d, s) not in seen]
print(f"steering: {len(STEER_T)} images x {len(combos)} remaining (layer, direction, signed size) combinations = {len(STEER_T)*len(combos)} forwards")
t0 = time.time()
for ci, (L, d, s) in enumerate(combos):
    vec = (s * DIRVEC[d][L]).astype(np.float32)
    for t in STEER_T: srows.append(dict(layer=L, dir=d, s=s, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t], edit=(L, "add", vec))))
    pd.DataFrame(srows).to_csv(sp, index=False); maybe_push('steering'); el = time.time() - t0
    print(f"  {ci+1}/{len(combos)} (L{L} {d} {s:+d}x) | {el/60:.1f} min | ~{el/(ci+1)*(len(combos)-ci-1)/60:.0f} min left", flush=True)
ST = pd.DataFrame(srows)

steering: 40 images x 100 remaining (layer, direction, signed size) combinations = 4000 forwards
  1/100 (L12 shift -1x) | 0.5 min | ~45 min left
  2/100 (L12 shift +1x) | 0.9 min | ~44 min left
  3/100 (L12 shift -4x) | 1.3 min | ~43 min left
  4/100 (L12 shift +4x) | 1.8 min | ~43 min left
  5/100 (L12 prior -1x) | 2.2 min | ~42 min left
  6/100 (L12 prior +1x) | 2.7 min | ~42 min left
  7/100 (L12 prior -4x) | 3.1 min | ~41 min left
  8/100 (L12 prior +4x) | 3.6 min | ~41 min left
  9/100 (L12 evid -1x) | 4.0 min | ~41 min left
  10/100 (L12 evid +1x) | 4.5 min | ~40 min left
  11/100 (L12 evid -4x) | 4.9 min | ~40 min left
  12/100 (L12 evid +4x) | 5.3 min | ~39 min left
  13/100 (L12 shuf -1x) | 5.8 min | ~39 min left
  14/100 (L12 shuf +1x) | 6.2 min | ~38 min left
  15/100 (L12 shuf -4x) | 6.7 min | ~38 min left
  16/100 (L12 shuf +4x) | 7.1 min | ~37 min left
  17/100 (L12 rand1 -1x) | 7.6 min | ~37 min left
  18/100 (L12 rand1 +1x) | 8.0 min | ~36 min left
  19/100 (L12 rand1 

In [12]:
# ===================== B2 / C analysis =====================
def pick(L, d, s): return ST[(ST.layer == L) & (ST.dir == d) & (ST.s == s)].set_index("t").delta
def eff_boot(L, d, a, n=300, seed=0):
    diff = ((pick(L, d, a) - pick(L, d, -a)) / 2).values; r = np.random.default_rng(seed); bs = [diff[r.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return diff.mean(), *np.percentile(bs, [2.5, 97.5])
b0hi = ST[(ST.dir == "base") & (ST.contrast >= 0.09)]; sw_base = b0hi[b0hi.side == "right"].delta.mean() - b0hi[b0hi.side == "left"].delta.mean()
def swing_ratio(L, d, s):
    x = ST[(ST.layer == L) & (ST.dir == d) & (ST.s == s) & (ST.contrast >= 0.09)]
    return (x[x.side == "right"].delta.mean() - x[x.side == "left"].delta.mean()) / sw_base
rs = []
for L in STEER_LAYERS:
    for d in DIRS:
        for a in AMPS:
            m_, lo, hi = eff_boot(L, d, a); rs.append(dict(layer=L, dir=d, size=a, effect=m_, lo=lo, hi=hi, frac_of_natural=m_ / NAT_HALF, swing_ratio_plus=swing_ratio(L, d, a), swing_ratio_minus=swing_ratio(L, d, -a)))
S2 = pd.DataFrame(rs); S2.to_csv(f"{OUT}/b2_steer_summary.csv", index=False)
say(f"\n[B2] steering. effect = (delta(+dir) - delta(-dir))/2 in logits. size 1x = norm of the natural half-jump of the count signal (same for every direction). natural behavioral half-jump = {NAT_HALF:.2f} logits")
for a in AMPS:
    say(f"  size {a}x:")
    for L in STEER_LAYERS: say(f"    layer {L}: " + " | ".join(f"{d} {S2[(S2.layer == L) & (S2.dir == d) & (S2['size'] == a)].effect.iloc[0]:+.2f}" for d in DIRS))
say("\nfull table (effect with 95% CI over images; swing ratios when adding +dir / -dir, 1 = evidence untouched):")
say(S2.round(2).to_string(index=False))
amax = max(AMPS); fig, ax = plt.subplots(1, 2, figsize=(12, 4)); cols = dict(zip(["shift", "prior", "evid", "shuf", "rand1"], ["C3", "C0", "C2", "C1", "gray"]))
for d in DIRS: x = S2[(S2.dir == d) & (S2["size"] == amax)]; ax[0].errorbar(x.layer, x.effect, yerr=[x.effect - x.lo, x.hi - x.effect], marker="o", label=d, color=cols[d], capsize=2)
ax[0].axhline(NAT_HALF, c="k", ls=":", label="natural half-jump"); ax[0].axhline(0, c="gray", lw=.5); ax[0].set_xlabel("layer"); ax[0].set_ylabel("effect on delta (logits)"); ax[0].legend(fontsize=7); ax[0].set_title(f"B2: steering at {amax}x natural size")
for d in DIRS: x = S2[(S2.dir == d) & (S2["size"] == amax)]; ax[1].plot(x.layer, (x.swing_ratio_plus + x.swing_ratio_minus) / 2, marker="o", label=d, color=cols[d])
ax[1].axhline(1, c="gray", lw=.5); ax[1].set_xlabel("layer"); ax[1].set_title("C: swing ratio under steering (1 = evidence untouched)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/b2_steer.png", dpi=130); plt.close()


[B2] steering. effect = (delta(+dir) - delta(-dir))/2 in logits. size 1x = norm of the natural half-jump of the count signal (same for every direction). natural behavioral half-jump = 0.33 logits
  size 1x:
    layer 12: shift +0.01 | prior +0.00 | evid +0.01 | shuf -0.01 | rand1 +0.00
    layer 14: shift -0.09 | prior +0.02 | evid +0.21 | shuf -0.06 | rand1 -0.01
    layer 16: shift +0.11 | prior +0.31 | evid +0.35 | shuf -0.30 | rand1 +0.00
    layer 20: shift +0.25 | prior +0.48 | evid +0.54 | shuf -0.48 | rand1 +0.00
    layer 24: shift +0.23 | prior +0.48 | evid +0.52 | shuf -0.45 | rand1 +0.00
  size 4x:
    layer 12: shift +0.02 | prior +0.02 | evid +0.02 | shuf -0.03 | rand1 +0.01
    layer 14: shift -0.31 | prior +0.10 | evid +0.73 | shuf -0.25 | rand1 -0.05
    layer 16: shift +0.44 | prior +1.26 | evid +1.41 | shuf -1.24 | rand1 +0.02
    layer 20: shift +1.01 | prior +1.95 | evid +2.20 | shuf -1.94 | rand1 +0.01
    layer 24: shift +0.91 | prior +1.91 | evid +2.10 | shuf -

In [13]:
# ===================== SUMMARY: paste everything printed by the previous cells back to Claude =====================
open(f"{OUT}/B_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push('final', force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")

[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (final) | server says: Ok
zip written: /kaggle/working/B_out.zip
